# 📘 Notebook 12: From Machine Learning to Neural Networks

### Course: *From Python to Deep Learning & Fine-Tuning*
**Instructor:** Ioannis Tsioulis

*Module D: Deep Learning · Notebook 1 of 4 in this module · (12 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain what a single **artificial neuron** computes
- Understand why **activation functions** make networks powerful
- Stack neurons into **layers** and a network, computing a **forward pass**
- Build a small neural network **from scratch in NumPy**, no deep-learning library
- See clearly how everything connects to the linear algebra and gradient descent you already know

> **Prerequisites:** Module C, and especially the NumPy linear algebra (Notebook 05) and gradient descent (Notebook 09).
>
> 🔭 **The big transition.** This notebook is the bridge from classical ML to deep learning. The good news: you already own every ingredient. A neuron is the linear layer $\mathbf{y} = W\mathbf{x} + \mathbf{b}$ from Notebook 05, followed by an activation function like the sigmoid from Notebook 10. Training it uses the gradient descent from Notebook 09. We are *assembling* familiar parts, not learning from zero.

> ℹ️ **Setup note.** This notebook uses only NumPy and Matplotlib: `import piplite; await piplite.install(['numpy','matplotlib'])`

## 1. The artificial neuron

### Intuition first
A biological neuron receives signals, combines them, and ‘fires’ if the combined signal is strong enough. The artificial version is a simple mathematical echo of this: it takes several inputs, forms a **weighted sum**, adds a **bias**, and passes the result through an **activation function**.

### The mathematics
For inputs $x_1, \dots, x_n$ with weights $w_1, \dots, w_n$ and bias $b$, the neuron computes:

$$z = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b = \mathbf{w} \cdot \mathbf{x} + b$$

and then applies an activation function $f$:

$$a = f(z)$$

That dot product $\mathbf{w}\cdot\mathbf{x}$ is exactly the operation you implemented in Notebook 05. A neuron is a dot product plus a bias plus a squashing function, nothing more.

In [ ]:
import numpy as np

def neuron(inputs, weights, bias, activation):
    z = np.dot(weights, inputs) + bias    # the familiar w . x + b
    return activation(z)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

x = np.array([0.5, 0.3, 0.9])
w = np.array([0.4, 0.7, 0.2])
b = 0.1

output = neuron(x, w, b, sigmoid)
print(f"Neuron output: {output:.4f}")

## 2. Activation functions: the source of power

### Why we cannot skip them
Here is a crucial fact: if every neuron were purely linear ($a = z$), then stacking many layers would still only produce a **linear** function overall, no more expressive than a single line. Real-world data is rarely linear. The **non-linear** activation function is what lets networks learn curves, corners, and complex boundaries. *Without non-linearity, depth is pointless.*

### The three you must know

In [ ]:
import matplotlib.pyplot as plt

def sigmoid(z): return 1 / (1 + np.exp(-z))
def tanh(z):    return np.tanh(z)
def relu(z):    return np.maximum(0, z)

z = np.linspace(-5, 5, 200)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, fn, name in zip(axes, [sigmoid, tanh, relu], ["Sigmoid", "Tanh", "ReLU"]):
    ax.plot(z, fn(z))
    ax.set_title(name); ax.grid(True); ax.axhline(0, color="gray", lw=0.5)
plt.tight_layout(); plt.show()

- **Sigmoid** $\sigma(z) = \frac{1}{1+e^{-z}}$ squashes to $(0,1)$, the same function you met in logistic regression (Notebook 10). Good for probabilities, but it weakens gradients in deep networks.
- **Tanh** squashes to $(-1, 1)$; zero-centred, often better than sigmoid for hidden layers.
- **ReLU** $f(z) = \max(0, z)$ is the modern default for hidden layers: cheap to compute and avoids the gradient-weakening problem. Most deep networks today use ReLU.

> 🧠 **Sigmoid, paid off.** When we introduced the sigmoid in Notebook 10, we promised it would return as a neural-network activation function. Here it is. The same small piece of mathematics serves two roles, a nice illustration of how few core ideas underpin the whole field.

## 3. From one neuron to a layer

### Stacking neurons
A **layer** is simply several neurons operating on the same inputs in parallel. Instead of a single weight vector, we now have a **weight matrix** $W$, one row per neuron, and a bias vector $\mathbf{b}$. The whole layer computes:

$$\mathbf{z} = W\mathbf{x} + \mathbf{b}, \qquad \mathbf{a} = f(\mathbf{z})$$

This is **exactly** the matrix-vector product $W\mathbf{x} + \mathbf{b}$ you computed in the final exercise of Notebook 05, now with an activation function wrapped around it. One matrix multiplication evaluates an entire layer of neurons at once, this is why vectorisation mattered.

In [ ]:
def layer(inputs, W, b, activation):
    z = W @ inputs + b        # one matmul = the whole layer (Notebook 05!)
    return activation(z)

# A layer of 3 neurons, each taking 2 inputs
np.random.seed(42)
W = np.random.randn(3, 2)     # shape (neurons, inputs)
b = np.random.randn(3)
x = np.array([1.0, 2.0])

a = layer(x, W, b, relu)
print("Layer output (3 neurons):", a.round(4))

## 4. Stacking layers into a network

### The forward pass
A network chains layers: the output of one layer becomes the input of the next. Running data through from input to output is called the **forward pass**. The layers between input and output are the **hidden layers**, ‘deep’ learning simply means having several of them.

Let us build a complete little network, input → hidden layer (ReLU) → output layer (sigmoid), by hand:

In [ ]:
class TinyNeuralNetwork:
    """A 2-layer network built from scratch in NumPy."""
    def __init__(self, n_inputs, n_hidden, n_outputs, seed=42):
        rng = np.random.default_rng(seed)   # reproducible weights
        # weights initialised small and random
        self.W1 = rng.standard_normal((n_hidden, n_inputs)) * 0.5
        self.b1 = np.zeros(n_hidden)
        self.W2 = rng.standard_normal((n_outputs, n_hidden)) * 0.5
        self.b2 = np.zeros(n_outputs)

    def forward(self, x):
        # hidden layer with ReLU
        z1 = self.W1 @ x + self.b1
        a1 = np.maximum(0, z1)
        # output layer with sigmoid
        z2 = self.W2 @ a1 + self.b2
        a2 = 1 / (1 + np.exp(-z2))
        return a2

net = TinyNeuralNetwork(n_inputs=3, n_hidden=4, n_outputs=1)
x = np.array([0.5, -0.2, 0.1])
print("Network prediction:", net.forward(x))

> 🔭 **PyTorch preview, paid off.** Notice the structure: a **class** holding the layers' parameters, with a `forward` method that defines how data flows. This is *precisely* the shape of a PyTorch model, `class MyNetwork(nn.Module)` with a `forward(self, x)` method, exactly as previewed in Notebook 04. In the next notebook we rewrite this same network in PyTorch and let it handle the training automatically.

> ⚠️ **Common pitfalls**
>
> - Shape mismatches are the number-one bug. For `W @ x`, the columns of `W` must equal the length of `x`. Print `.shape` at each layer while debugging, the golden rule from Notebook 05 applies throughout deep learning.
> - Forgetting the activation function turns the whole network back into a single linear function, however many layers it has.

## 5. What is still missing?

Our network makes predictions, but its weights are **random**, it has not *learned* anything yet. To learn, it needs to:
1. measure how wrong its predictions are (a **loss function**, like the MSE cost from Notebook 09);
2. compute how to adjust each weight to reduce that loss (the **gradients**);
3. update the weights by gradient descent (**exactly** the loop from Notebook 09, just with many more parameters).

Computing the gradients efficiently through many layers is done by an algorithm called **backpropagation**, the subject of the next notebook. The conceptual leap is small: it is the same ‘step downhill’ idea, applied layer by layer.

---
## ✏️ Exercises

### Exercise 1
Implement a single neuron with a **ReLU** activation that takes inputs `[2.0, -1.0]`, weights `[0.5, 0.5]`, and bias `0.0`. What is the output? Then change the bias to `-1.0` and observe the effect.

In [ ]:
import numpy as np
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def relu(z): return max(0, z)
x = np.array([2.0, -1.0]); w = np.array([0.5, 0.5])
print(relu(np.dot(w, x) + 0.0))   # z = 0.5 -> 0.5
print(relu(np.dot(w, x) - 1.0))   # z = -0.5 -> 0.0 (ReLU clips negatives)
```

*The bias shifts the threshold at which the neuron activates. With ReLU, a sufficiently negative bias switches the neuron 'off' (output 0).*
</details>

### Exercise 2
Using the `TinyNeuralNetwork` class, create a network with 2 inputs, 5 hidden units, and 1 output. Feed it the input `[1.0, 1.0]` and print the prediction. Then change the `seed` and confirm the prediction changes (because the random weights changed).

In [ ]:
import numpy as np
# (Re-run the TinyNeuralNetwork class cell first.)
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
net = TinyNeuralNetwork(2, 5, 1, seed=0)
print(net.forward(np.array([1.0, 1.0])))
net2 = TinyNeuralNetwork(2, 5, 1, seed=123)
print(net2.forward(np.array([1.0, 1.0])))
```

*Different seeds give different initial weights and therefore different (still-untrained) predictions. Training is what will make the weights meaningful.*
</details>

## 🔑 Key takeaways

- A **neuron** computes $f(\mathbf{w}\cdot\mathbf{x} + b)$, the dot product from Notebook 05 plus an activation.
- **Non-linear activations** (sigmoid, tanh, ReLU) are what make depth worthwhile; ReLU is the modern default.
- A **layer** is $f(W\mathbf{x}+\mathbf{b})$, one matrix multiply evaluates all its neurons at once.
- A **network** chains layers; running input to output is the **forward pass**.
- We built a real network in pure NumPy, its structure (a class with a `forward` method) is exactly PyTorch's.
- It cannot yet **learn**: that needs a loss, gradients, and backpropagation, the next notebook.

---
**Next:** *Notebook 13: Backpropagation & Training*, where the network finally learns, and we meet PyTorch.

---
*© Ioannis Tsioulis. *From Python to Deep Learning & Fine-Tuning*. Prepared for educational use.*